# **2. DATA PREPROCESSING**

## **2.1 LIBRARIES**

In [1]:
! pip install adlfs azure-storage-blob
! pip install holidays
! pip install xgboost

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.9/412.9 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.9/187.9 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.9/116.9 kB 10.7 MB/s eta 0:00:00


In [2]:
# STANDARD LIBRARY
import os
import io

# DATA MANIPULATION AND ANALYSIS
import pandas as pd
import numpy as np

# DATA VISUALISATION
import matplotlib.pyplot as plt
import seaborn as sns

# AZURE CLOUD STORAGE
from adlfs import AzureBlobFileSystem
from azure.storage.blob import BlobServiceClient

# HOLIDAYS
import holidays

In [3]:
# Correctly set pandas display option for max columns
pd.options.display.max_columns = None
pd.options.display.max_rows = None

## **2.2 LOADING DATASETS FROM AZURE BLOB STORAGE**

### **2.2.1 LOAD FROM AZURE BLOB STORAGE**

In [4]:
def load_csv_from_blob(blob_name):
    blob_client = container_client.get_blob_client(blob_name)
    stream = blob_client.download_blob()
    data_bytes = stream.readall()
    data_str = data_bytes.decode("utf-8")
    return pd.read_csv(io.StringIO(data_str))


# Connection configuration
AZURE_STORAGE_ACCOUNT = "researchprojectx24104515"
AZURE_STORAGE_KEY = "bxpexO6i+Hz6n1WiipTn+sTCuLPGMS1BogMERrIrHd16DpQ0GLfQ0R33yrSw4MxsDomq5yNMgw1o+AStlx/MjA=="
connection_string = f"DefaultEndpointsProtocol=https;AccountName={AZURE_STORAGE_ACCOUNT};AccountKey={AZURE_STORAGE_KEY};EndpointSuffix=core.windows.net"
blob_service_client = BlobServiceClient.from_connection_string(connection_string)


fs = AzureBlobFileSystem(
    account_name=AZURE_STORAGE_ACCOUNT,
    account_key=AZURE_STORAGE_KEY
)

In [ ]:
# Establish connection to cycle
CONTAINER_NAME = "preprocessed"
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

file_name = "cycle.csv"
cycle = load_csv_from_blob(file_name)
print(f"Loaded '{file_name}' with shape: {cycle.shape}")

Loaded 'cycle.csv' with shape: (54239, 7)


In [ ]:
# Establish connection to weather
CONTAINER_NAME = "preprocessed"
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

file_name = "weather.csv"
weather = load_csv_from_blob(file_name)
print(f"Loaded '{file_name}' with shape: {weather.shape}")

Loaded 'weather.csv' with shape: (54025, 8)


In [ ]:
# Establish connection to footfall
CONTAINER_NAME = "preprocessed"
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

file_name = "footfall.csv"
footfall = load_csv_from_blob(file_name)
print(f"Loaded '{file_name}' with shape: {weather.shape}")

Loaded 'footfall.csv' with shape: (54025, 8)


### **2.2.2 FIRST VIEW OF DATASETS**

In [ ]:
print("\nFootfall:")
print(footfall.shape)
footfall.head()


Footfall:
(53423, 18)


,date and time,FTF_Aston Quay Merged,FTF_Capel St Mary St Merged,FTF_College St Dame St Merged,FTF_Dame St Merged,FTF_Dolier St Burgh Quay Merged,FTF_Grafton St Monsoon Merged,FTF_Grafton St Nassau St Suffolk St,FTF_Henry St Merged,FTF_Mary St Merged,FTF_Oconnell St Princes St North Merged,FTF_Talbot St Guineys Merged,FTF_Westmoreland St East Fleet St Merged,FTF_Westmoreland St West Merged,hour,weekday,month,year
0,2019-01-01 00:00:00,0.0,238.0,0.0,0.0,0.0,140.0,0.0,597.0,163.0,1914.0,0.0,1670.0,1988.0,0,1,1,2019
1,2019-01-01 01:00:00,0.0,173.0,0.0,0.0,0.0,215.0,0.0,359.0,102.0,885.0,0.0,767.0,1270.0,1,1,1,2019
2,2019-01-01 02:00:00,0.0,121.0,0.0,0.0,0.0,210.0,0.0,317.0,63.0,984.0,0.0,642.0,1589.0,2,1,1,2019
3,2019-01-01 03:00:00,0.0,174.0,0.0,0.0,0.0,204.0,0.0,313.0,59.0,935.0,0.0,582.0,1534.0,3,1,1,2019
4,2019-01-01 04:00:00,0.0,82.0,0.0,0.0,0.0,88.0,0.0,172.0,46.0,390.0,0.0,143.0,610.0,4,1,1,2019


In [ ]:
footfall.tail()

,date and time,FTF_Aston Quay Merged,FTF_Capel St Mary St Merged,FTF_College St Dame St Merged,FTF_Dame St Merged,FTF_Dolier St Burgh Quay Merged,FTF_Grafton St Monsoon Merged,FTF_Grafton St Nassau St Suffolk St,FTF_Henry St Merged,FTF_Mary St Merged,FTF_Oconnell St Princes St North Merged,FTF_Talbot St Guineys Merged,FTF_Westmoreland St East Fleet St Merged,FTF_Westmoreland St West Merged,hour,weekday,month,year
53418,2025-02-03 19:00:00,2690.0,764.0,0.0,0.0,586.0,1206.0,215.0,0.0,494.0,993.0,0.0,0.0,766.0,19,0,2,2025
53419,2025-02-03 20:00:00,2206.0,1180.0,0.0,0.0,453.0,1301.0,137.0,0.0,323.0,744.0,0.0,0.0,566.0,20,0,2,2025
53420,2025-02-03 21:00:00,1561.0,406.0,0.0,0.0,337.0,544.0,103.0,0.0,174.0,552.0,0.0,0.0,441.0,21,0,2,2025
53421,2025-02-03 22:00:00,1483.0,236.0,0.0,0.0,302.0,513.0,139.0,0.0,118.0,393.0,0.0,0.0,403.0,22,0,2,2025
53422,2025-02-03 23:00:00,1241.0,341.0,0.0,0.0,165.0,286.0,78.0,0.0,56.0,259.0,0.0,0.0,277.0,23,0,2,2025


In [ ]:
print("\nWeather:")
print(weather.shape)
weather.head()


Weather:
(54025, 8)


,date and time,WTH_rain,WTH_temp,WTH_wetb,WTH_dewpt,WTH_vappr,WTH_rhum,WTH_msl
0,2019-01-01 00:00:00,0.0,9.6,7.8,5.6,9.1,76.0,1035.1
1,2019-01-01 01:00:00,0.0,8.6,7.1,5.3,8.9,79.0,1035.1
2,2019-01-01 02:00:00,0.0,8.3,6.9,5.3,8.9,81.0,1034.9
3,2019-01-01 03:00:00,0.0,9.1,7.6,5.8,9.2,79.0,1035.5
4,2019-01-01 04:00:00,0.0,9.2,7.7,5.9,9.3,79.0,1035.7


In [ ]:
weather.tail()

,date and time,WTH_rain,WTH_temp,WTH_wetb,WTH_dewpt,WTH_vappr,WTH_rhum,WTH_msl
54020,2025-02-28 20:00:00,0.0,6.0,3.6,-0.4,5.9,63.0,1028.6
54021,2025-02-28 21:00:00,0.0,4.4,2.7,-0.2,6.0,71.0,1029.0
54022,2025-02-28 22:00:00,0.0,4.1,2.7,0.4,6.3,76.0,1029.4
54023,2025-02-28 23:00:00,0.0,2.8,1.7,-0.2,6.0,80.0,1029.6
54024,2025-03-01 00:00:00,0.0,1.3,0.7,-0.5,5.9,87.0,1030.1


In [ ]:
print("\nCycle:")
print(cycle.shape)
cycle.head()


Cycle:
(54239, 7)


,date and time,CYC_Clontarf James Larkin Rd,CYC_Clontarf Pebble Beach Carpark,CYC_Grove Road Totem,CYC_North Strand Rd N B,CYC_Richmond Street Cyclists 1 Merged,CYC_Richmond Street Cyclists 2 Merged
0,2025-01-01 00:00:00,3.0,7.0,12.0,0.0,9.0,5.0
1,2025-01-01 01:00:00,2.0,2.0,17.0,0.0,1.0,1.0
2,2025-01-01 02:00:00,0.0,1.0,19.0,0.0,3.0,15.0
3,2025-01-01 03:00:00,0.0,1.0,7.0,0.0,3.0,12.0
4,2025-01-01 04:00:00,0.0,1.0,5.0,0.0,1.0,6.0


In [ ]:
cycle.tail()

,date and time,CYC_Clontarf James Larkin Rd,CYC_Clontarf Pebble Beach Carpark,CYC_Grove Road Totem,CYC_North Strand Rd N B,CYC_Richmond Street Cyclists 1 Merged,CYC_Richmond Street Cyclists 2 Merged
54234,2024-12-31 19:00:00,3.0,12.0,37.0,0.0,30.0,16.0
54235,2024-12-31 20:00:00,6.0,10.0,47.0,0.0,26.0,14.0
54236,2024-12-31 21:00:00,1.0,8.0,22.0,0.0,31.0,14.0
54237,2024-12-31 22:00:00,1.0,3.0,26.0,0.0,18.0,10.0
54238,2024-12-31 23:00:00,1.0,4.0,19.0,0.0,9.0,7.0


### **2.2.3 DATE AND TIME TO JOIN**

In [ ]:
cycle["date and time"] = pd.to_datetime(cycle["date and time"])
footfall["date and time"] = pd.to_datetime(footfall["date and time"])
weather["date and time"] = pd.to_datetime(weather["date and time"])

In [ ]:
weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54025 entries, 0 to 54024
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date and time  54025 non-null  datetime64[ns]
 1   WTH_rain       54025 non-null  float64       
 2   WTH_temp       54025 non-null  float64       
 3   WTH_wetb       54025 non-null  float64       
 4   WTH_dewpt      54025 non-null  float64       
 5   WTH_vappr      54025 non-null  float64       
 6   WTH_rhum       54025 non-null  float64       
 7   WTH_msl        54025 non-null  float64       
dtypes: datetime64[ns](1), float64(7)
memory usage: 3.3 MB


In [ ]:
cycle.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54239 entries, 0 to 54238
Data columns (total 7 columns):
 #   Column                                 Non-Null Count  Dtype         
---  ------                                 --------------  -----         
 0   date and time                          54239 non-null  datetime64[ns]
 1   CYC_Clontarf James Larkin Rd           54239 non-null  float64       
 2   CYC_Clontarf Pebble Beach Carpark      54239 non-null  float64       
 3   CYC_Grove Road Totem                   54239 non-null  float64       
 4   CYC_North Strand Rd N B                54239 non-null  float64       
 5   CYC_Richmond Street Cyclists 1 Merged  54239 non-null  float64       
 6   CYC_Richmond Street Cyclists 2 Merged  54239 non-null  float64       
dtypes: datetime64[ns](1), float64(6)
memory usage: 2.9 MB


In [ ]:
footfall.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53423 entries, 0 to 53422
Data columns (total 18 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   date and time                             53423 non-null  datetime64[ns]
 1   FTF_Aston Quay Merged                     53423 non-null  float64       
 2   FTF_Capel St Mary St Merged               53423 non-null  float64       
 3   FTF_College St Dame St Merged             53423 non-null  float64       
 4   FTF_Dame St Merged                        53423 non-null  float64       
 5   FTF_Dolier St Burgh Quay Merged           53423 non-null  float64       
 6   FTF_Grafton St Monsoon Merged             53423 non-null  float64       
 7   FTF_Grafton St Nassau St Suffolk St       53423 non-null  float64       
 8   FTF_Henry St Merged                       53423 non-null  float64       
 9   FTF_Mary St Merged          

### **2.2.4 PROCESSED DATAFRAME**

In [ ]:
def prepare_datetime_index(data, date_col="date and time", datetime_format=None, dayfirst=False):
    data = data.copy()
    data[date_col] = pd.to_datetime(data[date_col], format=datetime_format, dayfirst=dayfirst, errors="coerce")
    data.set_index(date_col, inplace=True)
    data.index.name = date_col
    data = data[~data.index.duplicated(keep="first")] # avoid duplicate datetimes
    data = data[~data.index.isna()]
    data = data.asfreq("h") # hourly frequencies
    return data

def merge_datetime(*dataframes, date_col="date and time", how="inner", verbose=False):
    if len(dataframes) < 2:
        raise ValueError("Two dataframes are needed to merge as minimum.")

    # Start with first DataFrame
    merged = dataframes[0].reset_index()

    for dataframe in dataframes[1:]:
        merged = pd.merge(
            merged,
            dataframe.reset_index(),
            on=date_col,
            how=how)

    # Restore the datetime index
    merged.set_index(date_col, inplace=True)

    if verbose:
        common_index = set(dataframes[0].index)
        for dataframe in dataframes[1:]:
            common_index = common_index.intersection(dataframe.index)
        print(f"Number of dates in common: {len(common_index)}")
        print(f"Merged DataFrame shape: {merged.shape}")
        pd.options.display.max_columns = None
        display(merged.head)

    return merged

cycle = prepare_datetime_index(cycle)
footfall = prepare_datetime_index(footfall)
weather = prepare_datetime_index(weather, datetime_format="%d/%m/%Y %H:%M", dayfirst=True)

data = merge_datetime(cycle, footfall, weather, verbose=True)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
def time_components(data):
  # Irish Holiday checked added to the dataframe
    ie_holidays = holidays.Ireland()
    datetime_index = data.index
    dates_series = pd.Series(datetime_index.date, index=data.index)

    # Time Components
    time_components = pd.DataFrame({
        "TEMP_hour": datetime_index.hour,
        "TEMP_weekday": datetime_index.dayofweek,
        "TEMP_month": datetime_index.month,
        "TEMP_year": datetime_index.year,
        "TEMP_holiday": dates_series.isin(ie_holidays).astype(int)
    }, index=data.index)

    data = pd.concat([data, time_components], axis=1)

    # Reset index
    data = data.reset_index()
    data.columns.name = None

    print(f"Shape: {data.shape}")
    print(f"Columns: {data.columns.tolist()}")

    return data

In [ ]:
data = time_components(data)

Shape: (53424, 36)
Columns: ['date and time', 'CYC_Clontarf James Larkin Rd', 'CYC_Clontarf Pebble Beach Carpark', 'CYC_Grove Road Totem', 'CYC_North Strand Rd N B', 'CYC_Richmond Street Cyclists 1 Merged', 'CYC_Richmond Street Cyclists 2 Merged', 'FTF_Aston Quay Merged', 'FTF_Capel St Mary St Merged', 'FTF_College St Dame St Merged', 'FTF_Dame St Merged', 'FTF_Dolier St Burgh Quay Merged', 'FTF_Grafton St Monsoon Merged', 'FTF_Grafton St Nassau St Suffolk St', 'FTF_Henry St Merged', 'FTF_Mary St Merged', 'FTF_Oconnell St Princes St North Merged', 'FTF_Talbot St Guineys Merged', 'FTF_Westmoreland St East Fleet St Merged', 'FTF_Westmoreland St West Merged', 'hour', 'weekday', 'month', 'year', 'WTH_rain', 'WTH_temp', 'WTH_wetb', 'WTH_dewpt', 'WTH_vappr', 'WTH_rhum', 'WTH_msl', 'TEMP_hour', 'TEMP_weekday', 'TEMP_month', 'TEMP_year', 'TEMP_holiday']


## **2.3 FEATURE ENGINEERING**

### **2.3.1 NEW FEATURES**

In [ ]:
# DYNAMIC WEATHER CONDITIONS

def weather_fts(data):

    # WEATHER INTERACTION

    # Mean of temperature in the last three hours recorded
    data["WTH_temp_3h"] = data["WTH_temp"].rolling(window=3).mean()

    # Rain intensitity bins (from no rain to very heavy)
    data["WTH_rain_int"] = pd.cut(
        data["WTH_rain"],
        bins=[-0.01, 0.0, 10.0, 30.0, 70.0, float("inf")],
        labels=[ 0, 1, 2, 3, 4 ]

        # the labels for the numbers are as follows ["no_rain", "slight", "moderate", "heavy", "very_heavy"]
    )

    # Probability of condensation
    # Dew point spread
    data["WTH_dew_spread"] = data["WTH_temp"] - data["WTH_dewpt"]
    data["WTH_prob_cond"] = np.where(
        (data["WTH_rhum"] >= 90) & (data["WTH_dew_spread"] <= 2), 1, 0)

    # WEATHER SEVERITY

    # Weather severity thresholds
    data["WTH_severe"] = 0
    data.loc[data["WTH_temp"] < 5, "WTH_severe"] += 1
    data.loc[data["WTH_rain"] > 4, "WTH_severe"] += 1

    return data

# DYNAMIC TIME CONDITIONS

def time_fts(data):
    data = data.copy()

    # TEMPORAL FEATURES
    data["TEMP_hour_sin"] = np.sin(2 * np.pi * data["TEMP_hour"] / 24)
    data["TEMP_hour_cos"] = np.cos(2 * np.pi * data["TEMP_hour"] / 24)
    data["TEMP_day_sin"] = np.sin(2 * np.pi * data["TEMP_weekday"] / 7)
    data["TEMP_day_cos"] = np.cos(2 * np.pi * data["TEMP_weekday"] / 7)
    data["TEMP_month_sin"] = np.sin(2 * np.pi * data["TEMP_month"] / 12)
    data["TEMP_month_cos"] = np.cos(2 * np.pi * data["TEMP_month"] / 12)

    # URBAN MOBILITY FEATURES
    # Check if the day is during the weekend
    data["TEMP_weekend"] = (data["TEMP_weekday"] >= 5).astype(int)

    # Morning peak between 7am and 9 am between Monday to Friday
    data["TEMP_mor_peak"] = (
        (data["TEMP_hour"] >= 7) &
        (data["TEMP_hour"] <= 9) &
        (data["TEMP_weekday"] < 5)
    ).astype(int)

    # Morning peak between 5pm (17:00) and 7pm (19:00) between Monday to Friday
    data["TEMP_eve_peak"] = (
        (data["TEMP_hour"] >= 17) &
        (data["TEMP_hour"] <= 19) &
        (data["TEMP_weekday"] < 5)
    ).astype(int)

    # Business hours during Monday to Friday (in this case 5 is Saturaday)
    data["TEMP_bus_hours"] = (
        (data["TEMP_hour"] >= 9) &
        (data["TEMP_hour"] <= 17) &
        (data["TEMP_weekday"] < 5)
    ).astype(int)

    return data

In [ ]:
data = weather_fts(data)
data = time_fts(data)

In [ ]:
#Dropping columns that will not be needed as sine and cosine was calculated
data = data.drop(columns=["hour", "weekday", "month", "year", "TEMP_hour", "TEMP_weekday", "TEMP_month"])

In [ ]:
data.shape

(53424, 44)

**Moving average of temperature**

The moving average technique allows to estimate trends, as a cycle of three-month pattern. As an indicator that the the past affect future events.

https://www.accaglobal.com/gb/en/student/exam-support-resources/fundamentals-exams-study-resources/f5/technical-articles/time-series.html


**Rain intensity**

The rain intensity has been classified according to the Rates of rainfall according to the National Center for Hydrology and Meteorology (NCHM) as categories but the threshold adapted to a more realistic to the Irish weather. They are as follows:
*   Slight rain: Less than 0.5 mm per hour.
*   Moderate rain: Greater than 0.5 mm per hour, but less than 4.0 mm per hour
*   Heavy rain: Greater than 4 mm per hour, but less than 8 mm per hour.
*   Very heavy rain: Greater than 8 mm per hour.

The zero values may be considered as no rain at all.

https://www.nchm.gov.bt/attachment/ckfinder/userfiles/files/Rainfall%20intensity%20classification.pdf


**Weather intensity**

Considering rough weather conditions as low temperature and super heavy rain.


**Probability of condensation**

The dew point spread reaches 2°C is the result between temperature minus dew point temperature. When the temperature and dew point spreads are almost equal, it may result in fog or condensation.

https://www.aopa.org/news-and-media/all-news/2003/march/pilot/wx-watch-dew-point-review

https://www.weather.gov/lmk/humidity#:~:text=Relative%20humidity%20(RH)%20(expressed,air%20at%20its%20current%20temperature.

https://library.fiveable.me/meteorology/unit-5/dew-point-relative-humidity/study-guide/RfxicYTAmshEsWlH

In [ ]:
data.head()

,date and time,CYC_Clontarf James Larkin Rd,CYC_Clontarf Pebble Beach Carpark,CYC_Grove Road Totem,CYC_North Strand Rd N B,CYC_Richmond Street Cyclists 1 Merged,CYC_Richmond Street Cyclists 2 Merged,FTF_Aston Quay Merged,FTF_Capel St Mary St Merged,FTF_College St Dame St Merged,FTF_Dame St Merged,FTF_Dolier St Burgh Quay Merged,FTF_Grafton St Monsoon Merged,FTF_Grafton St Nassau St Suffolk St,FTF_Henry St Merged,FTF_Mary St Merged,FTF_Oconnell St Princes St North Merged,FTF_Talbot St Guineys Merged,FTF_Westmoreland St East Fleet St Merged,FTF_Westmoreland St West Merged,WTH_rain,WTH_temp,WTH_wetb,WTH_dewpt,WTH_vappr,WTH_rhum,WTH_msl,TEMP_year,TEMP_holiday,WTH_temp_3h,WTH_rain_int,WTH_dew_spread,WTH_prob_cond,WTH_severe,TEMP_hour_sin,TEMP_hour_cos,TEMP_day_sin,TEMP_day_cos,TEMP_month_sin,TEMP_month_cos,TEMP_weekend,TEMP_mor_peak,TEMP_eve_peak,TEMP_bus_hours
0,2019-01-01 00:00:00,0.0,0.0,12.0,16.0,0.0,0.0,0.0,238.0,0.0,0.0,0.0,140.0,0.0,597.0,163.0,1914.0,0.0,1670.0,1988.0,0.0,9.6,7.8,5.6,9.1,76.0,1035.1,2019,0,NaN,0,4.0,0,0,0.000000,1.000000,0.781831,0.62349,0.5,0.866025,0,0,0,0
1,2019-01-01 01:00:00,0.0,0.0,17.0,14.0,0.0,0.0,0.0,173.0,0.0,0.0,0.0,215.0,0.0,359.0,102.0,885.0,0.0,767.0,1270.0,0.0,8.6,7.1,5.3,8.9,79.0,1035.1,2019,0,NaN,0,3.3,0,0,0.258819,0.965926,0.781831,0.62349,0.5,0.866025,0,0,0,0
2,2019-01-01 02:00:00,0.0,0.0,18.0,9.0,0.0,0.0,0.0,121.0,0.0,0.0,0.0,210.0,0.0,317.0,63.0,984.0,0.0,642.0,1589.0,0.0,8.3,6.9,5.3,8.9,81.0,1034.9,2019,0,8.833333,0,3.0,0,0,0.500000,0.866025,0.781831,0.62349,0.5,0.866025,0,0,0,0
3,2019-01-01 03:00:00,0.0,0.0,18.0,15.0,0.0,0.0,0.0,174.0,0.0,0.0,0.0,204.0,0.0,313.0,59.0,935.0,0.0,582.0,1534.0,0.0,9.1,7.6,5.8,9.2,79.0,1035.5,2019,0,8.666667,0,3.3,0,0,0.707107,0.707107,0.781831,0.62349,0.5,0.866025,0,0,0,0
4,2019-01-01 04:00:00,0.0,0.0,12.0,8.0,0.0,0.0,0.0,82.0,0.0,0.0,0.0,88.0,0.0,172.0,46.0,390.0,0.0,143.0,610.0,0.0,9.2,7.7,5.9,9.3,79.0,1035.7,2019,0,8.866667,0,3.3,0,0,0.866025,0.500000,0.781831,0.62349,0.5,0.866025,0,0,0,0


In [ ]:
data.columns

Index(['date and time', 'CYC_Clontarf James Larkin Rd',
       'CYC_Clontarf Pebble Beach Carpark', 'CYC_Grove Road Totem',
       'CYC_North Strand Rd N B', 'CYC_Richmond Street Cyclists 1 Merged',
       'CYC_Richmond Street Cyclists 2 Merged', 'FTF_Aston Quay Merged',
       'FTF_Capel St Mary St Merged', 'FTF_College St Dame St Merged',
       'FTF_Dame St Merged', 'FTF_Dolier St Burgh Quay Merged',
       'FTF_Grafton St Monsoon Merged', 'FTF_Grafton St Nassau St Suffolk St',
       'FTF_Henry St Merged', 'FTF_Mary St Merged',
       'FTF_Oconnell St Princes St North Merged',
       'FTF_Talbot St Guineys Merged',
       'FTF_Westmoreland St East Fleet St Merged',
       'FTF_Westmoreland St West Merged', 'WTH_rain', 'WTH_temp', 'WTH_wetb',
       'WTH_dewpt', 'WTH_vappr', 'WTH_rhum', 'WTH_msl', 'TEMP_year',
       'TEMP_holiday', 'WTH_temp_3h', 'WTH_rain_int', 'WTH_dew_spread',
       'WTH_prob_cond', 'WTH_severe', 'TEMP_hour_sin', 'TEMP_hour_cos',
       'TEMP_day_sin', 'TEMP_da

### **2.3.2 ZEROS AND NULL VALUES**

In [ ]:
# Percentage of 0 values per column
zero_percent = (data == 0).sum() / len(data) * 100

# Display nicely
print("Percentage of 0 values per column:\n")
print(zero_percent.sort_values(ascending=False))

Percentage of 0 values per column:

TEMP_holiday                                100.000000
TEMP_eve_peak                                91.071429
TEMP_mor_peak                                91.071429
WTH_rain                                     86.663297
WTH_rain_int                                 86.663297
WTH_severe                                   85.809748
TEMP_bus_hours                               73.214286
TEMP_weekend                                 71.428571
WTH_prob_cond                                69.644729
CYC_Clontarf James Larkin Rd                 43.437406
CYC_North Strand Rd N B                      42.844040
CYC_Clontarf Pebble Beach Carpark            41.954927
CYC_Richmond Street Cyclists 1 Merged        36.663297
CYC_Richmond Street Cyclists 2 Merged        35.004867
FTF_Aston Quay Merged                        25.131027
FTF_Dame St Merged                           24.852126
FTF_Westmoreland St East Fleet St Merged     22.970949
FTF_Henry St Merged          

In [ ]:
data = data.fillna(0)

In [ ]:
# Percentage of 0 values per column
zero_percent = (data == 0).sum() / len(data) * 100

# Display nicely
print("Percentage of 0 values per column:\n")
print(zero_percent.sort_values(ascending=False))

Percentage of 0 values per column:

TEMP_holiday                                100.000000
TEMP_eve_peak                                91.071429
TEMP_mor_peak                                91.071429
WTH_rain                                     86.663297
WTH_rain_int                                 86.663297
WTH_severe                                   85.809748
TEMP_bus_hours                               73.214286
TEMP_weekend                                 71.428571
WTH_prob_cond                                69.644729
CYC_Clontarf James Larkin Rd                 43.444894
CYC_North Strand Rd N B                      42.851527
CYC_Clontarf Pebble Beach Carpark            41.962414
CYC_Richmond Street Cyclists 1 Merged        36.670785
CYC_Richmond Street Cyclists 2 Merged        35.012354
FTF_Aston Quay Merged                        25.138515
FTF_Dame St Merged                           24.859614
FTF_Westmoreland St East Fleet St Merged     22.978437
FTF_Henry St Merged          

In [ ]:
data.head()

,date and time,CYC_Clontarf James Larkin Rd,CYC_Clontarf Pebble Beach Carpark,CYC_Grove Road Totem,CYC_North Strand Rd N B,CYC_Richmond Street Cyclists 1 Merged,CYC_Richmond Street Cyclists 2 Merged,FTF_Aston Quay Merged,FTF_Capel St Mary St Merged,FTF_College St Dame St Merged,FTF_Dame St Merged,FTF_Dolier St Burgh Quay Merged,FTF_Grafton St Monsoon Merged,FTF_Grafton St Nassau St Suffolk St,FTF_Henry St Merged,FTF_Mary St Merged,FTF_Oconnell St Princes St North Merged,FTF_Talbot St Guineys Merged,FTF_Westmoreland St East Fleet St Merged,FTF_Westmoreland St West Merged,WTH_rain,WTH_temp,WTH_wetb,WTH_dewpt,WTH_vappr,WTH_rhum,WTH_msl,TEMP_year,TEMP_holiday,WTH_temp_3h,WTH_rain_int,WTH_dew_spread,WTH_prob_cond,WTH_severe,TEMP_hour_sin,TEMP_hour_cos,TEMP_day_sin,TEMP_day_cos,TEMP_month_sin,TEMP_month_cos,TEMP_weekend,TEMP_mor_peak,TEMP_eve_peak,TEMP_bus_hours
0,2019-01-01 00:00:00,0.0,0.0,12.0,16.0,0.0,0.0,0.0,238.0,0.0,0.0,0.0,140.0,0.0,597.0,163.0,1914.0,0.0,1670.0,1988.0,0.0,9.6,7.8,5.6,9.1,76.0,1035.1,2019,0,0.000000,0,4.0,0,0,0.000000,1.000000,0.781831,0.62349,0.5,0.866025,0,0,0,0
1,2019-01-01 01:00:00,0.0,0.0,17.0,14.0,0.0,0.0,0.0,173.0,0.0,0.0,0.0,215.0,0.0,359.0,102.0,885.0,0.0,767.0,1270.0,0.0,8.6,7.1,5.3,8.9,79.0,1035.1,2019,0,0.000000,0,3.3,0,0,0.258819,0.965926,0.781831,0.62349,0.5,0.866025,0,0,0,0
2,2019-01-01 02:00:00,0.0,0.0,18.0,9.0,0.0,0.0,0.0,121.0,0.0,0.0,0.0,210.0,0.0,317.0,63.0,984.0,0.0,642.0,1589.0,0.0,8.3,6.9,5.3,8.9,81.0,1034.9,2019,0,8.833333,0,3.0,0,0,0.500000,0.866025,0.781831,0.62349,0.5,0.866025,0,0,0,0
3,2019-01-01 03:00:00,0.0,0.0,18.0,15.0,0.0,0.0,0.0,174.0,0.0,0.0,0.0,204.0,0.0,313.0,59.0,935.0,0.0,582.0,1534.0,0.0,9.1,7.6,5.8,9.2,79.0,1035.5,2019,0,8.666667,0,3.3,0,0,0.707107,0.707107,0.781831,0.62349,0.5,0.866025,0,0,0,0
4,2019-01-01 04:00:00,0.0,0.0,12.0,8.0,0.0,0.0,0.0,82.0,0.0,0.0,0.0,88.0,0.0,172.0,46.0,390.0,0.0,143.0,610.0,0.0,9.2,7.7,5.9,9.3,79.0,1035.7,2019,0,8.866667,0,3.3,0,0,0.866025,0.500000,0.781831,0.62349,0.5,0.866025,0,0,0,0


In [ ]:
data.describe()

,date and time,CYC_Clontarf James Larkin Rd,CYC_Clontarf Pebble Beach Carpark,CYC_Grove Road Totem,CYC_North Strand Rd N B,CYC_Richmond Street Cyclists 1 Merged,CYC_Richmond Street Cyclists 2 Merged,FTF_Aston Quay Merged,FTF_Capel St Mary St Merged,FTF_College St Dame St Merged,FTF_Dame St Merged,FTF_Dolier St Burgh Quay Merged,FTF_Grafton St Monsoon Merged,FTF_Grafton St Nassau St Suffolk St,FTF_Henry St Merged,FTF_Mary St Merged,FTF_Oconnell St Princes St North Merged,FTF_Talbot St Guineys Merged,FTF_Westmoreland St East Fleet St Merged,FTF_Westmoreland St West Merged,WTH_rain,WTH_temp,WTH_wetb,WTH_dewpt,WTH_vappr,WTH_rhum,WTH_msl,TEMP_year,TEMP_holiday,WTH_temp_3h,WTH_dew_spread,WTH_prob_cond,WTH_severe,TEMP_hour_sin,TEMP_hour_cos,TEMP_day_sin,TEMP_day_cos,TEMP_month_sin,TEMP_month_cos,TEMP_weekend,TEMP_mor_peak,TEMP_eve_peak,TEMP_bus_hours
count,53424,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.000000,53424.0,53424.000000,53424.000000,53424.000000,53424.000000,5.342400e+04,5.342400e+04,5.342400e+04,5.342400e+04,5.342400e+04,5.342400e+04,53424.000000,53424.000000,53424.000000,53424.000000
mean,2022-01-17 23:30:00,24.243898,33.268830,108.437425,27.062912,33.693246,33.708726,1430.093497,786.860924,482.710898,474.616614,649.422656,1233.176063,253.944613,2048.907888,397.992494,646.235381,936.822364,398.048873,688.820979,0.088168,10.588170,8.969137,7.237678,10.610173,80.565682,1013.066313,2021.553908,0.0,10.587789,3.350492,0.303553,0.141996,-1.848709e-17,-5.546128e-17,-2.021610e-17,-1.968410e-17,3.614338e-03,1.084934e-02,0.285714,0.089286,0.089286,0.267857
min,2019-01-01 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-5.100000,-5.400000,-7.500000,3.500000,21.000000,961.800000,2019.000000,0.0,-5.100000,0.000000,0.000000,0.000000,-1.000000e+00,-1.000000e+00,-9.749279e-01,-9.009689e-01,-1.000000e+00,-1.000000e+00,0.000000,0.000000,0.000000,0.000000
25%,2020-07-10 11:45:00,0.000000,0.000000,26.000000,0.000000,0.000000,0.000000,0.000000,67.000000,67.000000,1.000000,117.000000,40.000000,34.000000,22.000000,10.000000,78.000000,10.000000,4.000000,76.000000,0.000000,7.000000,5.800000,3.900000,8.100000,72.000000,1005.000000,2020.000000,0.0,7.033333,1.300000,0.000000,0.000000,-7.071068e-01,-7.071068e-01,-7.818315e-01,-9.009689e-01,-5.000000e-01,-5.000000e-01,0.000000,0.000000,0.000000,0.000000
50%,2022-01-17 23:30:00,2.000000,3.000000,81.000000,5.000000,10.000000,11.000000,747.000000,396.000000,334.000000,284.000000,585.000000,526.000000,165.000000,495.000000,104.000000,478.000000,297.000000,146.000000,544.000000,0.000000,10.600000,9.100000,7.400000,10.300000,83.000000,1014.400000,2022.000000,0.0,10.566667,2.700000,0.000000,0.000000,6.123234e-17,-6.123234e-17,0.000000e+00,-2.225209e-01,1.224647e-16,6.123234e-17,0.000000,0.000000,0.000000,0.000000
75%,2023-07-28 11:15:00,38.000000,52.000000,133.000000,36.000000,62.000000,59.000000,2502.000000,1161.000000,824.000000,765.000000,1037.000000,1936.000000,395.000000,3036.000000,589.000000,1070.000000,1663.000000,592.000000,1183.000000,0.000000,14.200000,12.300000,10.600000,12.800000,91.000000,1022.200000,2023.000000,0.0,14.233333,4.700000,1.000000,0.000000,7.071068e-01,7.071068e-01,7.818315e-01,6.234898e-01,5.000000e-01,8.660254e-01,1.000000,0.000000,0.000000,1.000000
max,2025-02-03 23:00:00,480.000000,628.000000,1152.000000,615.000000,314.000000,309.000000,15190.000000,9591.000000,4147.000000,4155.000000,5995.000000,11778.000000,2649.000000,23785.000000,7303.000000,3798.000000,10729.000000,5091.000000,5175.000000,12.800000,32.500000,21.500000,19.600000,22.700000

### **2.4 SAVE PREPROCESSED DATA TO BLOB STORAGE**

In [ ]:
def save_blob(data, filename):
    try:
        blob_name = f"{CONTAINER_NAME}/{filename}"
        csv_data = data.to_csv(index=False)
        with fs.open(blob_name, "w") as f:
            f.write(csv_data)
        print(f"Saved to {blob_name}")
    except Exception as e:
        print(f"Error saving data to blob storage: {str(e)}")
        return False

In [ ]:
CONTAINER_NAME = "preprocessed"

save_blob(data, "preprocessed.csv")

Saved to preprocessed/preprocessed.csv
